# Semana 2: janelas de 512 px do T2

Aplica a regra registrada em `prereg/t2_janelas.md` (commit `7ab6806`) aos 12 mapas do conjunto completo de `splits/t2_eventos.csv`. **Nenhum modelo roda neste notebook**, e não precisa de GPU.

1. Monta o Drive do mestrado e confere a lista do T2 pelo hash.
2. Corta as janelas, aplica as regras de entrada e de mínimo pontuável e grava as janelas que entram em `Mestrado/dados/worldfloods_v2/janelas`.
3. Grava `t2_janelas.csv` e `t2_janelas_info.json` em `Mestrado/resultados/semana2` e confere que cada pixel foi pontuado uma vez só.

Abra o Colab logada na conta do mestrado e rode na ordem. Se alguma célula der erro, pare e cole o erro no Claude.

In [ ]:
# CÉLULA 1. Drive, ambiente e lista do T2
!pip install -q "rasterio==1.4.3"

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, time, hashlib, shutil, datetime, platform
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window

MEUDRIVE = '/content/drive/MyDrive'
MESTRADO = os.path.join(MEUDRIVE, 'Mestrado')
assert not os.path.isdir(os.path.join(MEUDRIVE, 'DOC old PC')), (
    'Este é o Drive da conta antiga. Desconecte o ambiente, abra o Colab na conta do mestrado e rode de novo.')
assert os.path.isdir(MESTRADO), f'Não achei a pasta Mestrado. O Meu Drive desta conta tem: {sorted(os.listdir(MEUDRIVE))}'

DESTINO_WF = os.path.join(MESTRADO, 'dados', 'worldfloods_v2')
DADOS = '/content/wf2' if os.path.isdir('/content/wf2/test/S2') else DESTINO_WF
SAIDA = os.path.join(MESTRADO, 'resultados', 'semana2')
JANELAS_LOCAL = '/content/t2_janelas'
JANELAS_DRIVE = os.path.join(DESTINO_WF, 'janelas')

def sha256(p, bloco=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bloco), b''):
            h.update(b)
    return h.hexdigest()

SHA_LISTA = 'd0cd80324207e78fbc8fe02b9626610e914cc3b470bb8168d5be790235c5b892'
ARQ_LISTA = os.path.join(SAIDA, 't2_eventos.csv')
assert sha256(ARQ_LISTA) == SHA_LISTA, 'O t2_eventos.csv do Drive não é o registrado no repositório (commit 3b4e931). Cole esta saída no Claude.'
LISTA = pd.read_csv(ARQ_LISTA)
MAPAS = LISTA[LISTA.full_set].sort_values('map_id').reset_index(drop=True)

VERSOES = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
           'rasterio': rasterio.__version__, 'gdal': rasterio.__gdal_version__}
print(f'Lendo as cenas de {DADOS}')
print(f'{len(MAPAS)} mapas no conjunto completo, {int(MAPAS.core.sum())} no núcleo')
print(VERSOES)

In [ ]:
# CÉLULA 2. Cortar as janelas e aplicar a regra de prereg/t2_janelas.md (10 a 20 min)
TAM = 512
MAX_SEM_DADO = 0.01
MIN_PRINCIPAL = 26215                       # 10% de 512 x 512, arredondado para cima
MIN_SENSIBILIDADE = 1
INDICES = [1, 2, 3, 8, 11, 12]
NOMES = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']

def inicios(n):
    s = list(range(0, n - TAM + 1, TAM))
    if s[-1] + TAM < n:
        s.append(n - TAM)
    return s

for sub in ('S2', 'label'):
    os.makedirs(os.path.join(JANELAS_LOCAL, sub), exist_ok=True)
perfil_tif = dict(driver='GTiff', width=TAM, height=TAM, tiled=True, blockxsize=256, blockysize=256,
                  compress='deflate', predictor=2)

linhas, conferencia = [], []
t0 = time.time()
for m in MAPAS.itertuples():
    with rasterio.open(os.path.join(DADOS, 'test', 'gt', m.map_id + '.tif')) as g:
        gt = g.read()
    rotulo = np.full(gt.shape[1:], -1, np.int16)
    rotulo[gt[1] == 1] = 0
    rotulo[gt[1] == 2] = 1
    rotulo[gt[0] != 1] = -1
    H, W = rotulo.shape
    ja_pontuado = np.zeros((H, W), bool)
    pontuavel_cena = perdidos_sem_dado = 0
    with rasterio.open(os.path.join(DADOS, 'test', 'S2', m.map_id + '.tif')) as s:
        assert (s.height, s.width) == (H, W)
        for lin in inicios(H):
            for col in inicios(W):
                jan = Window(col, lin, TAM, TAM)
                x = s.read(indexes=list(range(1, 14)), window=jan)
                sem_dado = np.all(x == 0, axis=0)
                novo = ~ja_pontuado[lin:lin + TAM, col:col + TAM]
                ja_pontuado[lin:lin + TAM, col:col + TAM] = True
                r = rotulo[lin:lin + TAM, col:col + TAM].copy()
                perdidos_sem_dado += int(((r >= 0) & sem_dado & novo).sum())
                r[sem_dado | ~novo] = -1
                pontuavel_cena += int((r >= 0).sum())
                frac_sem_dado = float(sem_dado.mean())
                n = int((r >= 0).sum())
                entrada_ok = frac_sem_dado <= MAX_SEM_DADO
                jid = f'{m.map_id}_r{lin:05d}_c{col:05d}'
                linhas.append({'window_id': jid, 'map_id': m.map_id, 'event': m.event, 'core': bool(m.core),
                               'row_off': lin, 'col_off': col, 'frac_missing_input': frac_sem_dado,
                               'scored_pixels': n, 'water_pixels': int((r == 1).sum()), 'land_pixels': int((r == 0).sum()),
                               'kept_primary': entrada_ok and n >= MIN_PRINCIPAL,
                               'kept_sensitivity': entrada_ok and n >= MIN_SENSIBILIDADE})
                if entrada_ok and n >= MIN_SENSIBILIDADE:
                    tr = rasterio.windows.transform(jan, s.transform)
                    with rasterio.open(os.path.join(JANELAS_LOCAL, 'S2', jid + '.tif'), 'w', count=6, dtype='int16',
                                       crs=s.crs, transform=tr, **perfil_tif) as d:
                        d.write(x[INDICES].astype(np.int16))
                        d.descriptions = tuple(NOMES)
                    with rasterio.open(os.path.join(JANELAS_LOCAL, 'label', jid + '.tif'), 'w', count=1, dtype='int16',
                                       crs=s.crs, transform=tr, nodata=-1, **perfil_tif) as d:
                        d.write(r[None])
    esperado = int(((gt[1] != 0) & (gt[0] == 1)).sum())
    conferencia.append({'map_id': m.map_id, 'cobertura_completa': bool(ja_pontuado.all()),
                        'pontuados_nas_janelas': pontuavel_cena, 'perdidos_sem_dado': perdidos_sem_dado,
                        'rotulados_sem_nuvem_na_cena': esperado,
                        'clear_labeled_pixels_lista': int(m.clear_labeled_pixels)})
    print(f'{m.map_id}: {sum(1 for l in linhas if l["map_id"] == m.map_id)} janelas ({time.time() - t0:.0f} s)')

JAN = pd.DataFrame(linhas)
CONF = pd.DataFrame(conferencia)
print()
print(CONF.to_string(index=False))
assert CONF.cobertura_completa.all(), 'A grade não cobriu algum pixel. Cole esta saída no Claude.'
assert (CONF.pontuados_nas_janelas + CONF.perdidos_sem_dado == CONF.rotulados_sem_nuvem_na_cena).all(), (
    'A soma por janela não bate com a contagem da cena: algum pixel foi pontuado duas vezes ou nenhuma. Cole esta saída no Claude.')
print('\nCada pixel rotulado e sem nuvem foi pontuado exatamente uma vez.')

In [ ]:
# CÉLULA 3. Gravar o t2_janelas.csv, resumir e copiar as janelas para o Drive (5 a 10 min)
ARQ = os.path.join(SAIDA, 't2_janelas.csv')
JAN.to_csv(ARQ, index=False, float_format='%.6f')
SHA_JAN = sha256(ARQ)

resumo = dict(janelas=('window_id', 'size'), pixels=('scored_pixels', 'sum'))
print('Núcleo, por evento:')
nuc = JAN[JAN.core]
print(pd.concat({'principal': nuc[nuc.kept_primary].groupby('event').agg(**resumo),
                 'sensibilidade': nuc[nuc.kept_sensitivity].groupby('event').agg(**resumo)}, axis=1).to_string())
print()
print(f'Núcleo: {int(nuc.kept_primary.sum())} janelas no principal, {int(nuc.kept_sensitivity.sum())} na sensibilidade')
print(f'Conjunto completo: {int(JAN.kept_primary.sum())} janelas no principal, {int(JAN.kept_sensitivity.sum())} na sensibilidade')
print(f'Janelas fora pela regra de entrada (mais de 1% sem dado): {int((JAN.frac_missing_input > MAX_SEM_DADO).sum())}')

t0 = time.time()
shutil.copytree(JANELAS_LOCAL, JANELAS_DRIVE, dirs_exist_ok=True)
n_drive = sum(len(os.listdir(os.path.join(JANELAS_DRIVE, sub))) for sub in ('S2', 'label'))
assert n_drive >= 2 * int(JAN.kept_sensitivity.sum()), 'Faltaram arquivos de janela no Drive. Rode a célula de novo.'
print(f'Janelas copiadas para {JANELAS_DRIVE} em {(time.time() - t0) / 60:.1f} min')

INFO = {'gerado_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'regra': 'prereg/t2_janelas.md, commit 7ab6806',
        'lista_t2': {'arquivo': 'splits/t2_eventos.csv', 'sha256': SHA_LISTA},
        'tamanho': TAM, 'max_sem_dado': MAX_SEM_DADO, 'min_principal': MIN_PRINCIPAL, 'min_sensibilidade': MIN_SENSIBILIDADE,
        'bandas': dict(zip(NOMES, INDICES)), 'rotulo': {'agua': 1, 'terra': 0, 'ignorado': -1},
        'janelas': {'total': len(JAN), 'principal': int(JAN.kept_primary.sum()), 'sensibilidade': int(JAN.kept_sensitivity.sum())},
        'cada_pixel_uma_vez': True, 't2_janelas_csv_sha256': SHA_JAN, 'versoes': VERSOES, 'modelo_rodado': False}
json.dump(INFO, open(os.path.join(SAIDA, 't2_janelas_info.json'), 'w'), indent=2, ensure_ascii=False)
print(f'\nt2_janelas.csv  sha256 {SHA_JAN}')
print('Anexe no Claude o t2_janelas.csv e o t2_janelas_info.json de resultados/semana2.')